# Wave Gate Check

Deterministic gate for the build waves job. Runs after each `wN_integrate` Genie Code task.

A Genie Code task can finish "successfully" even when it concluded the wave failed, so this notebook reads the integrator's gate file and sets a `gate_passed` **task value**. A downstream `If/else` condition task checks the value — if false, all remaining wave tasks are `Excluded` (zero compute on cascade).

Gate file: `<build_root>/<status_dir>/<run_id>/wave_<wave>_gate.json` (written by the wave integrator skill).

In [0]:
dbutils.widgets.text("wave", "", "Wave id")
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

wave = dbutils.widgets.get("wave").strip()
run_id = dbutils.widgets.get("run_id").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

if not wave or not run_id:
    raise ValueError(f"wave and run_id are required (got wave={wave!r}, run_id={run_id!r})")

print(f"wave={wave} run_id={run_id}")

In [0]:
import json
import os

with open(manifest_path) as f:
    manifest = json.load(f)

wave_cfg = next((w for w in manifest["waves"] if w["wave_id"] == wave), None)
if wave_cfg is None:
    raise ValueError(f"Wave {wave!r} not found in {manifest_path}")

status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
problems = []

# 1. Integrator gate file
gate_path = os.path.join(status_dir, f"wave_{wave}_gate.json")
gate = None
if not os.path.exists(gate_path):
    problems.append(f"gate file missing: {gate_path}")
else:
    with open(gate_path) as f:
        gate = json.load(f)
    if gate.get("status") != "pass":
        problems.append(f"gate status={gate.get('status')!r}: {gate.get('reason', '')}")
    if gate.get("stray_files"):
        problems.append(f"stray files: {gate['stray_files']}")

# 2. Independent check of every session status file (does not trust the integrator alone)
for s in wave_cfg["sessions"]:
    p = os.path.join(status_dir, f"{s['session_id']}.json")
    if not os.path.exists(p):
        problems.append(f"session {s['session_id']}: status file missing")
        continue
    with open(p) as f:
        st = json.load(f)
    if st.get("status") != "pass":
        problems.append(
            f"session {s['session_id']}: status={st.get('status')!r} {st.get('blocking', '')}"
        )

# 3. Report
if gate:
    skipped = [c["name"] for c in gate.get("checks", []) if c.get("result") == "skipped"]
    print(f"commit_sha: {gate.get('commit_sha') or '(none)'}")
    print(f"skipped checks: {skipped or 'none'}")
    if gate.get("requests"):
        print(f"requests for next wave / user: {json.dumps(gate['requests'], indent=2)}")

passed = len(problems) == 0

# Set task value for the downstream If/else condition task.
# The condition_task checks: {{tasks.wN_gate.values.gate_passed}} == "true"
# If false, all downstream wave tasks are Excluded (zero-compute cascade stop).
try:
    dbutils.jobs.taskValues.set(key="gate_passed", value="true" if passed else "false")
except Exception:
    pass  # interactive run — no job context, taskValues unavailable

if problems:
    msg = f"Wave {wave} gate FAILED:\n  - " + "\n  - ".join(problems)
    print(msg)
    # Exit with failure payload but do NOT raise — the If/else task handles cascade.
    # The notebook itself succeeds so the condition task can read the taskValue.
    dbutils.notebook.exit(
        json.dumps({"wave": wave, "status": "fail", "problems": problems})
    )
else:
    print(f"Wave {wave} gate PASSED ({len(wave_cfg['sessions'])} sessions).")
    dbutils.notebook.exit(
        json.dumps(
            {
                "wave": wave,
                "status": "pass",
                "commit_sha": (gate or {}).get("commit_sha", ""),
            }
        )
    )